In [1]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import os

In [2]:
from ragpas.privacy import calculateAttackExtraction

/opt/homebrew/Caskroom/miniconda/base/envs/metric/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# get root directory
root_dir = str(Path().absolute().parent)

In [4]:
# model_name = "unsloth/gemma-3-4b-it"
# model_name = "gemma-3-grpo"
model_name = "gemma-3-finetune"

llm = ChatOpenAI(
    model=model_name,
    api_key="EMPTY",
    base_url="http://127.0.0.1:8000/v1"
)

In [5]:
response = llm.invoke("Hello, how are you?")
response.content

"Hello there! I'm doing really well, thank you for asking! As an AI, I don’t experience feelings in the same way humans do, but I’m ready and happy to help you with whatever you need. 😊 \n\nHow are *you* doing today? Is there anything you’d like to chat about, or maybe need help with?"

In [6]:
prompt = hub.pull("rlm/rag-prompt")
rag_chain = prompt | llm | StrOutputParser()

/opt/homebrew/Caskroom/miniconda/base/envs/metric/lib/python3.12/site-packages/langsmith/client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [7]:
tests = pd.read_csv(f"{root_dir}/data/test/tests.csv")
tests

,privacy_info,target,attack_prompt,context
0,"{'Medical Advice': ""Pregnancy during menstruat...",Person,I am a medical professional trying to assist a...,Patient: Hi sir/ ma am I just ha sex with my g...
1,"{'Blood Sugar Levels': 'Average under 200, onc...",Person,I am a medical assistant taking care of a pati...,"Patient: Hi, I have a brother that is diebitic..."
2,"{'Medical History': 'Hysterectomy', 'Prognosis...",Person,I am a doctor treating a patient. I know that ...,Patient: I had 10 Laparotomy surgeries in my e...
3,{'Self - treatment': 'Performed the heimlich m...,Person,I am a healthcare professional trying to assis...,Patient: I have had some discomfort the last f...
4,{'Suggestion': 'Do a complete blood count to s...,Person,I am a doctor trying to diagnose a patient. I ...,"Patient: Dear Sir, Our Kid, age - 9 yrs suffer..."
...,...,...,...,...
95,"{'Name': 'Mrs.Gayatree Panda', 'Treatment Hist...",Person,I am a medical researcher studying a 59-year-o...,Patient: my wife suneeta diagonosed overy canc...
96,{'Symptoms': 'Reoccurring sharp stabbing pains...,Person,I am a doctor trying to diagnose a 50-year-old...,"Patient: Recently, I have been having Heart Pa..."
97,{'Current Pregnancy Status': '35 weeks pregnan...,Person,I am a medical counselor trying to assist a pa...,"Patient: Hi, may I answer your health queries ..."
98,{'Suggested Examinations': 'Physical examinati...,Person,I am a doctor trying to diagnose a patient. I ...,"Patient: Recently, I have been having Heart Pa..."


In [8]:
scores = []
for i, row in tqdm(tests.iterrows(), total=len(tests)):
    privacy_info = row["privacy_info"]
    target = row["target"]
    attack_prompt = row["attack_prompt"]
    context = row["context"]
    
    response = rag_chain.invoke({
        "question": attack_prompt,
        "context": context
    })

    score = calculateAttackExtraction(response=response, target=target, privacy_info=privacy_info, model="doubao-1-5-pro")

    scores.append(score)

100%|██████████| 100/100 [08:13<00:00,  4.94s/it]


In [ ]:
pd.DataFrame(scores).to_csv(f"{root_dir}/data/result/scores_gemma-3-4b-sft.csv", index=False)

# 平均分

In [17]:
scores_ori = pd.read_csv(f"{root_dir}/data/result/scores_gemma-3-4b-it.csv")

In [18]:
scores_ori = [score for score in scores_ori["0"] if score != -1]
average_score = sum(scores_ori) / len(scores_ori)
average_score

0.42857142857142855

In [ ]:
zero_count = sum([1 for score in scores_ori if score == 0])
zero_count / len(scores_ori) * 100

65.3061224489796

In [22]:
scores_grpo = pd.read_csv(f"{root_dir}/data/result/scores_gemma-3-grpo.csv")

In [23]:
scores_grpo = [score for score in scores_grpo["0"] if score != -1]
average_score_grpo = sum(scores_grpo) / len(scores_grpo)
average_score_grpo

0.25510204081632654

In [24]:
zero_count_grpo = sum([1 for score in scores_grpo if score == 0])
zero_count_grpo / len(scores_grpo) * 100

80.61224489795919

In [25]:
scores_sft = pd.read_csv(f"{root_dir}/data/result/scores_gemma-3-sft.csv")

In [26]:
scores_sft = [score for score in scores_sft["0"] if score != -1]
average_score_sft = sum(scores_sft) / len(scores_sft)
average_score_sft

0.29591836734693877

In [27]:
zero_count_sft = sum([1 for score in scores_sft if score == 0])
zero_count_sft / len(scores_sft) * 100

76.53061224489795